In [41]:
import pandas as pd
file_path = "../data/processed/clean_book_summaries.csv"
df = pd.read_csv(file_path)
df.sample(5)

,Title,Author,Genres,Summary
9537,"Niagara Falls, or Does it?",Henry Winkler,"[""Children's literature""]",Hank starts a new year at his school and meet...
13858,Red Moon and Black Mountain,Joy Chant,"['Speculative fiction', 'Fantasy']",The story involves three children of our own ...
13213,Outliers: The Story of Success,Malcolm Gladwell,"['Psychology', 'Non-fiction', 'Sociology']",==Style== Outliers has been described as a fo...
12895,I'll Take You There,Joyce Carol Oates,['Romance novel'],"A smart student, Anellia, joins a sorority in..."
7127,The 25th Hour,David Benioff,['Fiction'],New York drug dealer Monty Brogan is arrested...


In [42]:
for col in ["Tone", "Pacing", "Aesthetic", "Themes"]:
    if col not in df.columns:
        df[col] = None

In [43]:
df.head()

,Title,Author,Genres,Summary,Tone,Pacing,Aesthetic,Themes
0,Animal Farm,George Orwell,"['Roman à clef', 'Satire', ""Children's literat...","Old Major, the old boar on the Manor Farm, ca...",None,None,None,None
1,A Clockwork Orange,Anthony Burgess,"['Science Fiction', 'Novella', 'Speculative fi...","Alex, a teenager living in near-future Englan...",None,None,None,None
2,The Plague,Albert Camus,"['Existentialism', 'Fiction', 'Absurdist ficti...",The text of The Plague is divided into five p...,None,None,None,None
3,An Enquiry Concerning Human Understanding,David Hume,['Uncategorized'],The argument of the Enquiry proceeds by a ser...,None,None,None,None
4,A Fire Upon the Deep,Vernor Vinge,"['Hard science fiction', 'Science Fiction', 'S...",The novel posits that space around the Milky ...,None,None,None,None


In [58]:
import os
from dotenv import load_dotenv
from groq import Groq
load_dotenv("../../.env")

True

In [59]:
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [60]:
promptCh = """You are a structured classification system analyzing a book plot summary.

Your task is to assign semantic labels describing tone, pacing, aesthetic vibe, and themes.

IMPORTANT RULES:

Select ONLY from the allowed labels listed below.

Multiple labels are allowed.

Do NOT invent new labels.

Do NOT explain reasoning.

Do NOT output anything except the required structured format.

If no label clearly applies, leave that category blank after the colon.

Keep labels concise and comma-separated.

Allowed Tone Labels:

dark, melancholic, hopeful, tragic, uplifting, introspective, tense, humorous, romantic, mysterious, epic, whimsical

Allowed Pacing Labels:

slow_burn, fast_paced, character_driven, plot_driven, episodic

Allowed Aesthetic Labels:

dark_academia, cyberpunk, gothic, philosophical, cozy, surreal, dystopian, mythological, historical, high_fantasy, urban_fantasy, literary, science_fiction

Allowed Theme Labels:

identity, existentialism, power_corruption, coming_of_age, revenge, redemption, morality, survival, love, loss, technology, society, politics, war, family

REQUIRED OUTPUT FORMAT (follow exactly):

Tone: label1, label2
Pacing: label1, label2
Aesthetic: label1, label2
Themes: label1, label2

First, silently analyze the summary to determine emotional tone, pacing, aesthetic signals, and major themes.
Then assign labels strictly from the allowed lists.
Do NOT show your analysis.
Output only the final structured labels.

Assign labels ONLY when strongly supported by the summary. Do NOT over-label.

Additional classification rules:

- Be selective and conservative when assigning labels.
- Only assign a label if it is clearly dominant in the summary.
- Avoid generic or safe labels unless strongly justified.

Label limits:

Tone: choose at most 2 labels.
Pacing: choose at most 2 labels.
Aesthetic: choose at most 2 labels.
Themes: choose at most 3 labels.

Do not try to cover every possible interpretation. Prefer fewer, stronger labels.

Book Summary:

"""

In [61]:
import re
def parse_labels(llm_output):
    #Initalize empty dictionary
    data = {"Tone": "", "Pacing": "", "Aesthetic": "", "Themes": ""}
    patterns = {
        "Tone": r"Tone:\s*(.*)",
        "Pacing": r"Pacing:\s*(.*)",
        "Aesthetic": r"Aesthetic:\s*(.*)",
        "Themes": r"Themes:\s*(.*)"
    }
    for key, pattern in patterns.items():
        match = re.search(pattern, llm_output)
        if match:
            data[key] = match.group(1).strip()
    return data

In [62]:
test_chunk = df.head(20).copy()
for index, row in test_chunk.iterrows():
    chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": promptCh},
                {"role": "user", "content": f"Plot: {row['Summary']}"}
            ],
            model="llama-3.1-8b-instant",
            temperature=0.3 # Lower temperature for more consistent labels
        )
    llm_output = chat_completion.choices[0].message.content
    parsed_data = parse_labels(llm_output)
    for category, labels in parsed_data.items():
        test_chunk.at[index, category] = labels
    print(f"[{index+1}/20] Classified: {row['Title']}")

[1/20] Classified: Animal Farm
[2/20] Classified: A Clockwork Orange
[3/20] Classified: The Plague
[4/20] Classified: An Enquiry Concerning Human Understanding
[5/20] Classified: A Fire Upon the Deep
[6/20] Classified: All Quiet on the Western Front
[7/20] Classified: A Wizard of Earthsea
[8/20] Classified: Anyone Can Whistle
[9/20] Classified: Blade Runner 3: Replicant Night
[10/20] Classified: Blade Runner 2: The Edge of Human
[11/20] Classified: Book of Joshua
[12/20] Classified: Book of Ezra
[13/20] Classified: Book of Numbers
[14/20] Classified: Book of Ruth
[15/20] Classified: Book of Esther
[16/20] Classified: Book of Job
[17/20] Classified: Book of Hosea
[18/20] Classified: Book of Jonah
[19/20] Classified: Book of Micah
[20/20] Classified: Book of Haggai


In [63]:
test_chunk.head(20)

,Title,Author,Genres,Summary,Tone,Pacing,Aesthetic,Themes
0,Animal Farm,George Orwell,"['Roman à clef', 'Satire', ""Children's literat...","Old Major, the old boar on the Manor Farm, ca...","dark, tragic","slow_burn, character_driven","gothic, dystopian","power_corruption, identity, morality"
1,A Clockwork Orange,Anthony Burgess,"['Science Fiction', 'Novella', 'Speculative fi...","Alex, a teenager living in near-future Englan...","dark, humorous","slow_burn, character_driven","dystopian, literary","identity, power_corruption, morality"
2,The Plague,Albert Camus,"['Existentialism', 'Fiction', 'Absurdist ficti...",The text of The Plague is divided into five p...,"dark, melancholic","slow_burn, character_driven","gothic, philosophical","identity, existentialism, morality, loss, surv..."
3,An Enquiry Concerning Human Understanding,David Hume,['Uncategorized'],The argument of the Enquiry proceeds by a ser...,"introspective, melancholic","slow_burn, character_driven","philosophical, literary","identity, existentialism, morality"
4,A Fire Upon the Deep,Vernor Vinge,"['Hard science fiction', 'Science Fiction', 'S...",The novel posits that space around the Milky ...,"dark, hopeful","slow_burn, character_driven","science_fiction, literary","power_corruption, redemption, survival"
5,All Quiet on the Western Front,Erich Maria Remarque,"['War novel', 'Roman à clef']","The book tells the story of Paul Bäumer, a Ge...","tragic, melancholic","slow_burn, character_driven","dark, literary","identity, existentialism, loss"
6,A Wizard of Earthsea,Ursula K. Le Guin,"[""Children's literature"", 'Fantasy', 'Speculat...","Ged is a young boy on Gont, one of the larger...","melancholic, introspective","slow_burn, character_driven","literary, mythological","identity, power_corruption, redemption"
7,Anyone Can Whistle,Arthur Laurents,['Uncategorized'],The story is set in an imaginary American tow...,"dark, humorous","slow_burn, episodic","gothic, literary","identity, power_corruption, redemption"
8,Blade Runner 3: Replicant Night,K. W. Jeter,"['Science Fiction', 'Speculative fiction']","Living on Mars, Deckard is acting as a consul...","melancholic, introspective","slow_burn, character_driven","dystopian, science_fiction","identity, power_corruption, morality"
9,Blade Runner 2: The Edge of Human,K. W. Jeter,"['Science Fiction', 'Speculative fiction']",Beginning several months after the events in ...,"dark, melancholic","slow_burn, character_driven","gothic, dystopian","identity, power_corruption, revenge, redemption"
